# TP53 - exploration

In [1]:
import functools as ft
import pandas as pd
import os
import pickle as pkl
import re

In [2]:
# wd
os.chdir("/mnt/c/Users/Dell/Alma Mater Studiorum Università di Bologna/Bioinformatics Seràgnoli - Documenti/EVERGREEN_Bioinformatica/CoMMpass_portal_downloads/IA22/")
outDir="/mnt/c/Users/Dell/Desktop/git_projects/TP53/"

In [3]:
# reading the df
mmrf_df=pd.read_csv("expression_estimates_transcript_based/MMRF_CoMMpass_IA22_salmon_transcriptUnstrandedIgFiltered_counts.tsv", sep="\t", index_col=0)
mmrf_df

,MMRF_1021_1_BM_CD138pos,MMRF_1024_1_BM_CD138pos,MMRF_1024_3_BM_CD138pos,MMRF_1024_2_BM_CD138pos,MMRF_1029_1_BM_CD138pos,MMRF_1030_1_BM_CD138pos,MMRF_1030_3_BM_CD138pos,MMRF_1030_4_BM_CD138pos,MMRF_1030_5_BM_CD138pos,MMRF_1031_1_BM_CD138pos,...,MMRF_2921_2_BM_CD138pos,MMRF_2924_1_BM_CD138pos,MMRF_2926_1_BM_CD138pos,MMRF_2936_1_PB_CD138pos,MMRF_2938_1_BM_CD138pos,MMRF_2939_1_BM_CD138pos,MMRF_2940_1_BM_CD138pos,MMRF_2941_1_BM_CD138pos,MMRF_2946_1_BM_CD138pos,MMRF_2947_1_BM_CD138pos
Transcript,,,,,,,,,,,,,,,,,,,,,
ENST00000000233,953.080,1481.820,1508.156,831.736,1384.857,693.857,874.605,3674.100,4526.460,860.448,...,2613.685,1223.449,1187.770,7156.645,1965.706,2782.523,3587.295,3614.036,2245.019,3032.207
ENST00000000412,4429.823,6050.341,6362.648,3754.320,10887.414,3934.778,5214.255,12300.601,5994.598,5334.391,...,16303.343,6170.538,3633.265,10975.095,11323.552,22478.751,16331.601,4057.584,17269.558,17270.860
ENST00000000442,600.068,2497.376,4562.165,2093.354,1843.254,2675.086,1681.132,3280.532,3352.618,1022.878,...,1670.557,740.718,1636.100,2158.735,2046.878,4077.976,13199.099,7505.208,4082.440,8434.376
ENST00000001008,2359.285,3770.100,2821.231,2147.167,2831.345,1611.774,4341.578,14503.645,9585.649,2131.782,...,7477.418,2304.555,1635.291,2738.629,8214.866,5775.432,4874.765,1278.660,7699.509,7051.009
ENST00000001146,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,57.000,0.000,1.141,0.000,0.000,0.000,0.000,0.000,0.000,24.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENST00000673604,0.000,0.000,0.000,0.000,14.982,0.000,0.000,2.884,0.000,73.820,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,10.850
ENST00000673615,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,108.513,0.000,0.000,0.000,1.000
ENST00000673617,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


### Overall dataframe
224.779 transcripts expression (encoded with Ensembl ID) for 933 samples (defined as MMRF_*_BM_CD138pos). Additionally, there are 806 pts.

In [4]:
# dimensions
mmrf_df.shape

(224779, 933)

In [5]:
# columns
mmrf_df.columns

Index(['MMRF_1021_1_BM_CD138pos', 'MMRF_1024_1_BM_CD138pos',
       'MMRF_1024_3_BM_CD138pos', 'MMRF_1024_2_BM_CD138pos',
       'MMRF_1029_1_BM_CD138pos', 'MMRF_1030_1_BM_CD138pos',
       'MMRF_1030_3_BM_CD138pos', 'MMRF_1030_4_BM_CD138pos',
       'MMRF_1030_5_BM_CD138pos', 'MMRF_1031_1_BM_CD138pos',
       ...
       'MMRF_2921_2_BM_CD138pos', 'MMRF_2924_1_BM_CD138pos',
       'MMRF_2926_1_BM_CD138pos', 'MMRF_2936_1_PB_CD138pos',
       'MMRF_2938_1_BM_CD138pos', 'MMRF_2939_1_BM_CD138pos',
       'MMRF_2940_1_BM_CD138pos', 'MMRF_2941_1_BM_CD138pos',
       'MMRF_2946_1_BM_CD138pos', 'MMRF_2947_1_BM_CD138pos'],
      dtype='object', length=933)

In [6]:
# NA?
mmrf_df.isnull().values.any()

False

In [7]:
# list of unique MMRF IDs
IDs=mmrf_df.columns.str.split("_[\\d+]_[PBM]+_CD138pos$").str.join("").unique()
IDs.size

806

In [8]:
mmrf_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 224779 entries, ENST00000000233 to ENST00000673621
Columns: 933 entries, MMRF_1021_1_BM_CD138pos to MMRF_2947_1_BM_CD138pos
dtypes: float64(933)
memory usage: 1.6+ GB


`pydeseq2` requires that `counts` - MMRF_df in this case - is build with only integers! Rounding.

In [9]:
mmrf_df=mmrf_df.round(0).astype("int")
mmrf_df.to_csv(os.path.join(outDir, "mmrf_IA22_transcripts_df.tsv"), sep="\t")

### Metadata
To perform differential expression analysis (DEA) DeSeq2 requires the following inputs:

* A pandas dataframe which shape is [samples x transcripts], containing expression values (non-negative values); 
* A pandas metadata dataframe which shape is [samples x number of variables], containing sample annotations that will be used to split the data in cohorts.

Additionally, we extract all clinical info that may be relevant to the overall analysis.

#### Clinical metadata
Three tables retrieved from MMRF consortium:

* PER PATIENT: pts general info. 
* PER VISIT: biochemical data of pts collected during visits. 
* PER TRT: treatment and therapy info per pt.

In [10]:
# pts info metadata
md_pts=pd.read_csv("clinical_flat_files/CoMMpass_IA22_FlatFiles/MMRF_CoMMpass_IA22_PER_PATIENT.tsv", sep="\t")[["PUBLIC_ID","D_PT_age","D_PT_gender","D_PT_iss"]]
md_pts

,PUBLIC_ID,D_PT_age,D_PT_gender,D_PT_iss
0,MMRF_1014,64,2,1.0
1,MMRF_1017,75,1,1.0
2,MMRF_1024,79,2,2.0
3,MMRF_1038,69,1,3.0
4,MMRF_1033,64,1,1.0
...,...,...,...,...
1138,MMRF_2846,61,2,3.0
1139,MMRF_2847,57,1,3.0
1140,MMRF_2848,45,2,2.0
1141,MMRF_2851,62,2,1.0


Filtering pts visit metadata as:

* baseline results.

In [11]:
# pts visit metadata
md_visit=pd.read_csv("clinical_flat_files/CoMMpass_IA22_FlatFiles/MMRF_CoMMpass_IA22_PER_PATIENT_VISIT.tsv", sep="\t", dtype=str)[["PUBLIC_ID","VJ_INTERVAL","SS_HYPERCALCEMIA","SS_RENALINSUFFIC","SS_ANEMIAHEMOGLO","SS_BONELESIONSLY","ST_PLASMACYTOMAN","IC_INVOLVEDFREEL","D_LAB_serum_m_protein","D_LAB_urine_24hr_m_protein"]]
md_visit=md_visit.loc[md_visit["VJ_INTERVAL"]=="Baseline"]
md_visit

,PUBLIC_ID,VJ_INTERVAL,SS_HYPERCALCEMIA,SS_RENALINSUFFIC,SS_ANEMIAHEMOGLO,SS_BONELESIONSLY,ST_PLASMACYTOMAN,IC_INVOLVEDFREEL,D_LAB_serum_m_protein,D_LAB_urine_24hr_m_protein
2,MMRF_1014,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.2,NaN
19,MMRF_1017,Baseline,NaN,NaN,NaN,checked,NaN,NaN,2.2,NaN
29,MMRF_1024,Baseline,NaN,checked,checked,NaN,NaN,NaN,2.6,0
59,MMRF_1038,Baseline,checked,NaN,NaN,checked,NaN,NaN,3.9,0.018
82,MMRF_1033,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.9,1.426
...,...,...,...,...,...,...,...,...,...,...
22233,MMRF_2846,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.84,NaN
22241,MMRF_2847,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,5.380000114,NaN
22250,MMRF_2848,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,5.309999943,1.812999964
22260,MMRF_2851,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000024


Filtering pts treatment metadata as:

* having line of therapy equal to 1; 
* by therapy last day - to have a single line per pts containing all treatments performed in line 1.

In [12]:
# pts treatment metadata
md_trt=pd.read_csv("clinical_flat_files/CoMMpass_IA22_FlatFiles/MMRF_CoMMpass_IA22_STAND_ALONE_TRTRESP.tsv", sep="\t", encoding="ISO-8859-1")[["PUBLIC_ID","line","thername","therstdy","therendy"]]
md_trt=md_trt.loc[md_trt["line"]==1].sort_values('therendy', ascending=False).drop_duplicates(['PUBLIC_ID'])
md_trt

,PUBLIC_ID,line,thername,therstdy,therendy
1455,MMRF_1173,1,Bortezomib/Lenalidomide/Dexamethasone + Lenali...,1,2976
2616,MMRF_1120,1,Bortezomib/Lenalidomide/Dexamethasone + Bortez...,1,2973
1303,MMRF_1773,1,Bortezomib/Lenalidomide/Dexamethasone + Bortez...,1,2962
83,MMRF_1395,1,Bortezomib/Lenalidomide/Dexamethasone + Lenali...,1,2948
618,MMRF_1054,1,Lenalidomide/Dexamethasone + Lenalidomide/Pred...,1,2948
...,...,...,...,...,...
3135,MMRF_2078,1,Bortezomib,1,1
3922,MMRF_2754,1,Carfilzomib/Lenalidomide/Dexamethasone,1,1
2683,MMRF_1778,1,Bortezomib/Dexamethasone,1,1
3900,MMRF_2739,1,Carfilzomib/Cyclophosphamide/Dexamethasone,1,1


For each metadata df, there are 1143 unique `PUBLIC_ID`. Thus, not all pts have RNA-seq data associated (unique `PUBLIC_ID`s from `tp53_df` dataframe are 806).

In [13]:
md_pts['PUBLIC_ID'].unique().size

1143

Merging all clinical metadata dfs to obtain a single df per pt.

In [14]:
md_clinical_df=ft.reduce(lambda left, right: pd.merge(left, right, on='PUBLIC_ID'), [md_pts, md_visit, md_trt])
md_clinical_df

,PUBLIC_ID,D_PT_age,D_PT_gender,D_PT_iss,VJ_INTERVAL,SS_HYPERCALCEMIA,SS_RENALINSUFFIC,SS_ANEMIAHEMOGLO,SS_BONELESIONSLY,ST_PLASMACYTOMAN,IC_INVOLVEDFREEL,D_LAB_serum_m_protein,D_LAB_urine_24hr_m_protein,line,thername,therstdy,therendy
0,MMRF_1014,64,2,1.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.2,NaN,1,Bortezomib/Dexamethasone + Bortezomib/Lenalido...,1,1157
1,MMRF_1017,75,1,1.0,Baseline,NaN,NaN,NaN,checked,NaN,NaN,2.2,NaN,1,Lenalidomide/Dexamethasone,1,508
2,MMRF_1024,79,2,2.0,Baseline,NaN,checked,checked,NaN,NaN,NaN,2.6,0,1,Thalidomide/Melphalan/Prednisone + Lenalidomid...,1,582
3,MMRF_1038,69,1,3.0,Baseline,checked,NaN,NaN,checked,NaN,NaN,3.9,0.018,1,Lenalidomide/Dexamethasone,1,1738
4,MMRF_1033,64,1,1.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.9,1.426,1,Lenalidomide/Dexamethasone,1,123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,MMRF_2846,61,2,3.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,2.84,NaN,1,Carfilzomib/Cyclophosphamide/Dexamethasone,1,121
1168,MMRF_2847,57,1,3.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,5.380000114,NaN,1,Carfilzomib/Lenalidomide/Dexamethasone + Carfi...,1,131
1169,MMRF_2848,45,2,2.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,5.309999943,1.812999964,1,Carfilzomib/Lenalidomide/Dexamethasone + Carfi...,1,154
1170,MMRF_2851,62,2,1.0,Baseline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000024,1,Carfilzomib/Cyclophosphamide/Dexamethasone,1,122


In [15]:
md_clinical_df.to_csv(os.path.join(outDir, "mmrf_clinical_data.tsv"), sep="\t")

#### TP53 metadata
Creating a metadata df containing Ensembl IDs for TP53 isoforms (binary).

In [16]:
# defining dataframe data
tp53_cols=["isoform", "ensembl_ID"]
tp53_data=[("p53alpha", "ENST00000269305"), ("p53beta", "ENST00000420246"), ("p53gamma", "ENST00000455263"),
           ("delta40p53alpha", "ENST00000619485"), ("delta40p53beta", "ENST00000622645"), 
           ("delta133p53alpha", "ENST00000504937"), ("delta160p53alpha", "ENST00000619186"), ("delta133p53beta", "ENST00000510385")]

# creating df
tp53_df=pd.DataFrame(data=tp53_data, columns=tp53_cols)
tp53_df

,isoform,ensembl_ID
0,p53alpha,ENST00000269305
1,p53beta,ENST00000420246
2,p53gamma,ENST00000455263
3,delta40p53alpha,ENST00000619485
4,delta40p53beta,ENST00000622645
5,delta133p53alpha,ENST00000504937
6,delta160p53alpha,ENST00000619186
7,delta133p53beta,ENST00000510385


In [17]:
tp53_df.to_csv(os.path.join(outDir, "tp53_isoforms.tsv"), sep="\t")

**CHECK**: filtering the transcripts df to keep only those ENSEMBL IDs that correspond to tp53's isoforms.

In [18]:
mmrf_df[mmrf_df.index.isin(tp53_df.ensembl_ID)]

,MMRF_1021_1_BM_CD138pos,MMRF_1024_1_BM_CD138pos,MMRF_1024_3_BM_CD138pos,MMRF_1024_2_BM_CD138pos,MMRF_1029_1_BM_CD138pos,MMRF_1030_1_BM_CD138pos,MMRF_1030_3_BM_CD138pos,MMRF_1030_4_BM_CD138pos,MMRF_1030_5_BM_CD138pos,MMRF_1031_1_BM_CD138pos,...,MMRF_2921_2_BM_CD138pos,MMRF_2924_1_BM_CD138pos,MMRF_2926_1_BM_CD138pos,MMRF_2936_1_PB_CD138pos,MMRF_2938_1_BM_CD138pos,MMRF_2939_1_BM_CD138pos,MMRF_2940_1_BM_CD138pos,MMRF_2941_1_BM_CD138pos,MMRF_2946_1_BM_CD138pos,MMRF_2947_1_BM_CD138pos
Transcript,,,,,,,,,,,,,,,,,,,,,
ENST00000269305,596.129,2349.101,2380.068,1409.214,2002.701,1244.701,3278.659,4718.687,1103.093,2330.068,...,3630.624,64.343,612.970,12494.432,3138.669,4422.630,345.896,193.046,4364.360,4890.428
ENST00000420246,397.856,108.809,321.993,146.308,517.376,194.796,425.903,737.214,840.814,200.061,...,1476.522,342.058,205.344,1566.328,1212.534,276.959,3749.292,1376.038,1728.157,2653.986
ENST00000455263,0.000,0.000,26.617,14.190,0.000,0.000,34.640,0.000,41.532,0.000,...,169.270,0.000,0.000,170.441,141.578,24.904,128.662,0.000,128.374,117.789
ENST00000504937,0.000,0.000,76.922,30.699,15.719,14.920,49.747,80.740,86.804,11.436,...,232.934,256.688,14.422,179.088,258.592,71.459,750.440,178.447,106.209,307.427
ENST00000510385,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


Isoforms $\small\Delta40p53\alpha$, $\small\Delta40p53\beta$, and $\small\Delta160p53\alpha$ are not present in CoMMpass data.

#### Technical metadata
Building a custom - but required - metadata df necessary to subset the complete df for downstream normalization step. The column `rep` depicts the replicate number for that unique ID.

In [19]:
md_df=mmrf_df.columns.str.extract("_(\\d)_")
md_df.index=mmrf_df.columns
md_df.columns=["rep"]
md_df

,rep
MMRF_1021_1_BM_CD138pos,1
MMRF_1024_1_BM_CD138pos,1
MMRF_1024_3_BM_CD138pos,3
MMRF_1024_2_BM_CD138pos,2
MMRF_1029_1_BM_CD138pos,1
...,...
MMRF_2939_1_BM_CD138pos,1
MMRF_2940_1_BM_CD138pos,1
MMRF_2941_1_BM_CD138pos,1
MMRF_2946_1_BM_CD138pos,1


In [20]:
md_df.to_csv(os.path.join(outDir, "metadata.tsv"), sep="\t")